In [1]:
# notebooks/05_xgboost_model.ipynb — CELL 1

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    average_precision_score
)

from xgboost import XGBClassifier

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import shap
import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# -----------------------------------
# CREATE FOLDERS
# -----------------------------------
os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# -----------------------------------
# LOAD DATA
# -----------------------------------
df = pd.read_csv('../data/fd_loans_engineered.csv')

# -----------------------------------
# LOAD LR METRICS
# -----------------------------------
lr_m = json.load(
    open('../models/lr_metrics.json')
)

FEATURES = lr_m['features']

TARGET = 'defaulted'

X = df[FEATURES]
y = df[TARGET]

# -----------------------------------
# TRAIN TEST SPLIT
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -----------------------------------
# XGBOOST MODEL
# -----------------------------------
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=(
        (y_train == 0).sum()
        /
        (y_train == 1).sum()
    ),

    eval_metric='auc',

    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# -----------------------------------
# TRAIN MODEL
# -----------------------------------
xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# -----------------------------------
# PREDICTIONS
# -----------------------------------
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

y_pred_xgb = xgb.predict(X_test)

# -----------------------------------
# METRICS
# -----------------------------------
auc_xgb = roc_auc_score(
    y_test,
    y_proba_xgb
)

auc_pr_xgb = average_precision_score(
    y_test,
    y_proba_xgb
)

print(f"\nXGBoost AUC-ROC: {auc_xgb:.4f}")

print(f"XGBoost AUC-PR: {auc_pr_xgb:.4f}")

print("\nCLASSIFICATION REPORT:\n")

print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=['No Default', 'Default']
    )
)

# -----------------------------------
# SHAP ANALYSIS
# -----------------------------------
print("\nComputing SHAP values...")

explainer = shap.TreeExplainer(xgb)

X_sample = X_test.iloc[:500]

shap_values = explainer.shap_values(X_sample)

# -----------------------------------
# SHAP FEATURE IMPORTANCE PLOT
# -----------------------------------
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=FEATURES,
    plot_type='bar',
    show=False
)

plt.title(
    'XGBoost — Feature Importance (SHAP)',
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    '../plots/shap_xgboost.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/shap_xgboost.png")

# -----------------------------------
# SAVE MODEL
# -----------------------------------
joblib.dump(
    xgb,
    '../models/xgb_model.pkl'
)

# -----------------------------------
# SAVE METRICS
# -----------------------------------
json.dump(
    {
        'auc_roc': round(auc_xgb, 4),
        'auc_pr': round(auc_pr_xgb, 4),
        'features': FEATURES,
        'lr_auc': lr_m['auc_roc']
    },
    open('../models/xgb_metrics.json', 'w'),
    indent=2
)

print(f"\nSaved: models/xgb_model.pkl")

print(f"XGBoost AUC: {auc_xgb:.4f}")

D:\fd loan risk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[0]	validation_0-auc:0.65573
[50]	validation_0-auc:0.67225
[100]	validation_0-auc:0.66900
[150]	validation_0-auc:0.66457
[200]	validation_0-auc:0.66178
[250]	validation_0-auc:0.65858
[299]	validation_0-auc:0.65361

XGBoost AUC-ROC: 0.6536
XGBoost AUC-PR: 0.1315

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

  No Default       0.94      0.73      0.82      7333
     Default       0.13      0.45      0.21       667

    accuracy                           0.71      8000
   macro avg       0.54      0.59      0.52      8000
weighted avg       0.87      0.71      0.77      8000


Computing SHAP values...
Saved: plots/shap_xgboost.png

Saved: models/xgb_model.pkl
XGBoost AUC: 0.6536
